# CLIP Image Embedding Generation

In this notebook, we use OpenAI's CLIP model to convert product images into feature vectors (embeddings). These embeddings will later be used to perform semantic similarity search and recommend the most relevant products based on user text queries.

In [1]:
import torch
import clip
from PIL import Image
import numpy as np
import pandas as pd
import os

# Check PyTorch Installation

In [2]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.12.1+cpu
False


In [3]:
import clip

print("CLIP imported successfully!")

CLIP imported successfully!


# Load Pretrained CLIP Model

Load OpenAI's pretrained CLIP model and select the appropriate device (CPU or GPU).

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess = clip.load("ViT-B/32", device=device)

print("Device:", device)

Device: cpu


# Load Cleaned Dataset

In [5]:
df = pd.read_csv("../data/processed/cleaned_electronics_products.csv")

df.head()

,Unnamed: 0,name,main_category,sub_category,image,link,ratings,no_of_ratings,discount_price,actual_price
0,0,"Redmi 10 Power (Power Black, 8GB RAM, 128GB St...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/81eM15lVcJ...,https://www.amazon.in/Redmi-Power-Black-128GB-...,4.0,965,"₹10,999","₹18,999"
1,1,"OnePlus Nord CE 2 Lite 5G (Blue Tide, 6GB RAM,...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/71AvQd3Vzq...,https://www.amazon.in/OnePlus-Nord-Lite-128GB-...,4.3,"113,956","₹18,999","₹19,999"
2,2,OnePlus Bullets Z2 Bluetooth Wireless in Ear E...,"tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/51UhwaQXCp...,https://www.amazon.in/Oneplus-Bluetooth-Wirele...,4.2,"90,304","₹1,999","₹2,299"
3,3,"Samsung Galaxy M33 5G (Mystique Green, 6GB, 12...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/81I3w4J6yj...,https://www.amazon.in/Samsung-Mystique-Storage...,4.1,"24,863","₹15,999","₹24,999"
4,4,"OnePlus Nord CE 2 Lite 5G (Black Dusk, 6GB RAM...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/71V--WZVUI...,https://www.amazon.in/OnePlus-Nord-Black-128GB...,4.3,"113,956","₹18,999","₹19,999"


# Check Downloaded Images

In [6]:
image_folder = "../data/images"

print("Total downloaded images:", len(os.listdir(image_folder)))

Total downloaded images: 9599


# Generate Embedding for One Image

In [7]:
image_path = os.path.join(image_folder, "0.jpg")

image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)

with torch.no_grad():
    embedding = model.encode_image(image)

print(embedding.shape)

torch.Size([1, 512])


# Generate Image Embeddings

Convert every downloaded product image into a 512-dimensional embedding using the pretrained CLIP model.

In [8]:
embeddings = []
image_names = []

In [9]:
image_files = sorted(os.listdir(image_folder))

print("Total Images Found:", len(image_files))

Total Images Found: 9599


# Generate Embeddings for All Downloaded Images

In [10]:
embeddings = []
image_names = []
failed_images = []

In [11]:
from tqdm import tqdm

image_files = sorted(os.listdir(image_folder))

for image_name in tqdm(image_files, desc="Generating Embeddings"):

    try:
        image_path = os.path.join(image_folder, image_name)

        image = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)

        with torch.no_grad():
            embedding = model.encode_image(image)

        embeddings.append(embedding.cpu().numpy().flatten())
        image_names.append(image_name)

    except Exception as e:
        failed_images.append({
            "image_name": image_name,
            "error": str(e)
        })

Generating Embeddings:   0%|          | 0/9599 [00:00<?, ?it/s]

Generating Embeddings: 100%|██████████| 9599/9599 [14:24<00:00, 11.11it/s] 


In [12]:
embeddings = np.array(embeddings)
print("Embedding Shape:", embeddings.shape)

Embedding Shape: (9599, 512)


In [13]:
os.makedirs("../data/embeddings", exist_ok=True)

np.save("../data/embeddings/image_embeddings.npy", embeddings)

In [14]:
mapping_df = pd.DataFrame({
    "image_name": image_names
})

mapping_df.to_csv(
    "../data/embeddings/image_mapping.csv",
    index=False
)

In [15]:
failed_df = pd.DataFrame(failed_images)

failed_df.to_csv(
    "../data/embeddings/failed_embeddings.csv",
    index=False
)

print(f"Failed Images: {len(failed_images)}")

Failed Images: 0


In [16]:
import numpy as np

embeddings = np.load("../data/embeddings/image_embeddings.npy")

print(embeddings.shape)

(9599, 512)
